1. agent개념

In [7]:
import openai 

client = openai.OpenAI()

PROMPT= """
I have the following functions in my system. 

`get_weather`
`get_currency`
`get_news`

All of them receive the name of a country as an argument. (i.e get_news('Spain'))

Please answer with the name of the function that you would like me to run. 

Please say nothing else, just the name of the function with the arguments. 

Answer the following question:

What is the weather in Greece?

"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    n = 10,
    messages=[{"role": "user", "content": PROMPT}]
)

response

ChatCompletion(id='chatcmpl-Dpn5m5QNsebZoxDDsjACQOp7HNDiq', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="get_weather('Greece')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)), Choice(finish_reason='stop', index=1, logprobs=None, message=ChatCompletionMessage(content="get_weather('Greece')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)), Choice(finish_reason='stop', index=2, logprobs=None, message=ChatCompletionMessage(content="get_weather('Greece')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)), Choice(finish_reason='stop', index=3, logprobs=None, message=ChatCompletionMessage(content="get_weather('Greece')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)), Choice(finish_reason='stop', index=4, logprobs=None, message=ChatComple

In [9]:
message = response.choices[0].message.content

message

"get_weather('Greece')"

2. 메모리

In [13]:
import openai 

client = openai.OpenAI()
messages = []

In [14]:
def call_ai(): 
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    
    print(f"AI: {message}")


In [15]:
while True: 
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        
        call_ai()


User: my name is jiwhi
AI: Nice to meet you, Jiwhi! How can I assist you today?
User: i live in seoul
AI: That's great! Seoul is a vibrant city with a rich culture and history. Do you have any favorite places or activities in Seoul? Or is there something specific you'd like to know or discuss about the city?
User: what is my first question i asked you and what is the closest country to where i am live?
AI: Your first question was about your name, Jiwhi, and you mentioned that you live in Seoul. The closest country to South Korea is North Korea, which shares a border with it. Other nearby countries include Japan, which is a short distance across the sea, and China, which is to the west. If you have more questions or need information about anything else, feel free to ask!


3. agent 

In [25]:
import openai 
import json

client = openai.OpenAI()
messages = []

In [26]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [27]:
def get_weather(city):
    return "33 degres celcius."

FUNCTION_MAP = {
    "get_weather": get_weather
} 

In [28]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage): 
    if message.tool_calls: 
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in message.tool_calls
            ] 
        } )

        for tool_call in message.tool_calls: 
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try : 
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}


            function_to_run = FUNCTION_MAP[function_name]
            result = function_to_run(**arguments)

            print(f"Calling function: {function_name} with arguments: {arguments} for a result of {result}")
            
            messages.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id,
                "name": function_name
            })

        call_ai()
        
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


In [29]:
def call_ai(): 
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)
    

In [30]:
while True: 
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        
        call_ai()

User: my name is jihwi
AI: Nice to meet you, Jihwi! How can I assist you today?
User: what is my name?
AI: Your name is Jihwi. How can I assist you further?
User: how is the weather of spain?
Calling function: get_weather with arguments: {'city': 'Spain'} for a result of 33 degres celcius.
AI: The weather in Spain is currently 33 degrees Celsius. If you need more specific information about a particular city or region in Spain, just let me know!


json.load 테스트

In [ ]:
a= '{"city": "Spain"}'

b = json.loads(a)

**b

get_weather(city='Spain')

{'city': 'Spain'}